# 02 Exploratory data analysis

Goal: understand how customers behave so we know which features matter for
next quarter fuel litres, next quarter non-fuel spend and the Opportunity label.

Sections:

1. Activity over time
2. What a typical customer looks like
3. Categories
4. When people shop
5. The targets at past cutoffs
6. What predicts next quarter
7. Takeaways

In [ ]:
# setup
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from sasol_clv.pipelines.data_processing.nodes import clean_transactions
from sasol_clv.pipelines.feature_engineering.labels import build_labels

RAW = Path("../data/01_raw")

# one simple look for every chart
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({
    "figure.figsize": (10, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e6e5e1",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
})

In [ ]:
# load and clean
raw = pd.read_csv(
    RAW / "train.csv",
    dtype={"ID": str, "TransactionId": str, "LocationId": str, "StockItemId": str},
    keep_default_na=False,
    na_values=[""],
)
cfg = json.load(open(RAW / "label_config.json"))
S_FUEL = cfg["normalization"]["CLV_fuel"]
S_NONFUEL = cfg["normalization"]["CLV_nonfuel"]

tx = clean_transactions(raw, cfg)
tx["month"] = tx["DateTimeZA"].dt.to_period("M").dt.to_timestamp()
tx.shape

## 1. Activity over time

In [ ]:
# monthly totals, we drop March 2024 because it only starts on the 19th
m = tx[tx["month"] >= "2024-04-01"]
monthly = pd.DataFrame({
    "customers": m.groupby("month")["ID"].nunique(),
    "baskets": m.groupby("month")["TransactionId"].nunique(),
    "fuel_litres": m[m["is_fuel_litres"]].groupby("month")["Quantity"].sum(),
    "nonfuel_rand": m[m["is_nonfuel"]].groupby("month")["TotalExclAmount"].sum(),
})

# one chart per measure, they are on very different scales
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
for ax, col, title in zip(
    axes.ravel(),
    ["customers", "baskets", "fuel_litres", "nonfuel_rand"],
    ["Active customers", "Baskets", "Fuel litres", "Non-fuel spend (R, excl VAT)"],
):
    ax.plot(monthly.index, monthly[col], color=BLUE, linewidth=2, marker="o", markersize=4)
    ax.set_title(title, loc="left")
    ax.set_ylim(bottom=0)
fig.tight_layout()
monthly

Monthly active customers stay fairly flat (roughly 3 000 to 3 400), but non-fuel spend almost doubles from 2024 to 2025.
The number of eligible customers grows at every cutoff because more people pass the 3 basket rule.
Fuel dips a little in Dec and Jan. The test quarter is Dec to Feb, so festive season effects matter.

## 2. What a typical customer looks like

In [ ]:
per_cust = pd.DataFrame({
    "baskets": tx.groupby("ID")["TransactionId"].nunique(),
    "fuel_litres": tx[tx["is_fuel_litres"]].groupby("ID")["Quantity"].sum(),
    "nonfuel_rand": tx[tx["is_nonfuel"]].groupby("ID")["TotalExclAmount"].sum(),
    "sites": tx.groupby("ID")["LocationId"].nunique(),
}).fillna(0)
per_cust.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).T

In [ ]:
# spend is very skewed, which is why the targets are logged
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].hist(np.log1p(per_cust["baskets"]), bins=50, color=BLUE)
axes[0].set_title("log(1 + baskets)", loc="left")
axes[1].hist(np.log1p(per_cust["fuel_litres"].clip(lower=0)), bins=50, color=BLUE)
axes[1].set_title("log(1 + fuel litres)", loc="left")
axes[2].hist(np.log1p(per_cust["nonfuel_rand"].clip(lower=0)), bins=50, color=BLUE)
axes[2].set_title("log(1 + non-fuel rand)", loc="left")
fig.tight_layout()

In [ ]:
# how many customers never buy anything in the shop
print("share with zero non-fuel spend:", (per_cust["nonfuel_rand"] <= 0).mean().round(3))
print("share with zero fuel litres   :", (per_cust["fuel_litres"] <= 0).mean().round(3))

## 3. Categories

In [ ]:
inscope = tx[tx["in_scope"]]
cat = pd.DataFrame({
    "rand": inscope.groupby("ItemCategoryLevel2")["TotalExclAmount"].sum(),
    "customers": inscope.groupby("ItemCategoryLevel2")["ID"].nunique(),
    "baskets": inscope.groupby("ItemCategoryLevel2")["TransactionId"].nunique(),
}).sort_values("customers", ascending=False)
cat["share_of_customers"] = (cat["customers"] / tx["ID"].nunique()).round(3)
cat

In [ ]:
# share of customers that ever bought each category (fuel left out, almost everyone buys it)
c = cat.drop(index="Fuel").sort_values("share_of_customers")
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(c.index, c["share_of_customers"], color=BLUE, height=0.7)
ax.set_title("Share of customers that ever bought the category", loc="left")
ax.grid(axis="y", visible=False)
fig.tight_layout()

Fuel dominates in both rows and rand. Beverages and Food & beverage are the main shop categories.
Most shop categories are bought by a minority of customers, which leaves lots of room for "New category adoption".

## 4. When people shop

In [ ]:
b = tx.drop_duplicates("TransactionId")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
hour = b["DateTimeZA"].dt.hour.value_counts().sort_index()
axes[0].bar(hour.index, hour.values, color=BLUE, width=0.8)
axes[0].set_title("Baskets by hour of day", loc="left")
dow = b["DateTimeZA"].dt.day_name().value_counts().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
)
axes[1].bar(dow.index.str[:3], dow.values, color=BLUE, width=0.7)
axes[1].set_title("Baskets by day of week", loc="left")
fig.tight_layout()

## 5. The targets at past cutoffs

In [ ]:
# rebuild labels at every quarter start we can fully observe
cutoffs = ["2024-06-01", "2024-09-01", "2024-12-01", "2025-03-01", "2025-06-01", "2025-09-01"]
labels = []
for cutoff in cutoffs:
    lab = build_labels(tx, cutoff, cfg)
    lab["cutoff"] = cutoff
    labels.append(lab)
labels = pd.concat(labels, ignore_index=True)
labels["z_fuel"] = np.log1p(labels["y_fuel"].clip(lower=0)) / S_FUEL
labels["z_nonfuel"] = np.log1p(labels["y_nonfuel"].clip(lower=0)) / S_NONFUEL

labels.groupby("cutoff").agg(
    customers=("ID", "size"),
    zero_fuel=("y_fuel", lambda s: (s <= 0).mean()),
    zero_nonfuel=("y_nonfuel", lambda s: (s <= 0).mean()),
    mean_z_fuel=("z_fuel", "mean"),
    mean_z_nonfuel=("z_nonfuel", "mean"),
).round(3)

In [ ]:
# the transformed targets have a big spike at zero, the rest is roughly bell shaped
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].hist(labels["z_fuel"], bins=60, color=BLUE)
axes[0].set_title("CLV_fuel (transformed)", loc="left")
axes[1].hist(labels["z_nonfuel"], bins=60, color=BLUE)
axes[1].set_title("CLV_nonfuel (transformed)", loc="left")
fig.tight_layout()

In [ ]:
# label mix per cutoff, look at the Dec cutoff since the test window is Dec to Feb
mix = pd.crosstab(labels["Opportunity"], labels["cutoff"], normalize="columns").round(3)
mix.sort_values("2025-09-01", ascending=False)

Four labels carry most of the weight: Stable, Existing-category growth: Fuel, Inactivity and Existing-category growth: Other.
Weighted F1 rewards getting these big classes right, so they matter most.
Inactivity climbs from about 7% at the Jun 2024 cutoff to about 27% at Sep 2025, as the pool of older customers who drift away grows.

## 6. What predicts next quarter

In [ ]:
# last quarter fuel versus next quarter fuel at the Sep 2025 cutoff
cutoff = pd.Timestamp("2025-09-01")
prev = tx[(tx["DateTimeZA"] >= cutoff - pd.DateOffset(months=3)) & (tx["DateTimeZA"] < cutoff)]
prev_fuel = prev[prev["is_fuel_litres"]].groupby("ID")["Quantity"].sum()
lab = labels[labels["cutoff"] == "2025-09-01"].set_index("ID")
x = np.log1p(prev_fuel.reindex(lab.index).fillna(0).clip(lower=0)) / S_FUEL

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x, lab["z_fuel"], s=8, alpha=0.25, color=BLUE, edgecolors="none")
ax.plot([0, 4], [0, 4], color=GREY, linewidth=1, linestyle="--")
ax.set_xlabel("last quarter (transformed)")
ax.set_ylabel("next quarter (transformed)")
ax.set_title("Fuel: last quarter vs next quarter", loc="left")
print("correlation:", np.corrcoef(x, lab["z_fuel"])[0, 1].round(3))

In [ ]:
# recency is the strongest signal for Inactivity
last_seen = tx[tx["DateTimeZA"] < cutoff].groupby("ID")["DateTimeZA"].max()
lab["recency_days"] = (cutoff - last_seen.reindex(lab.index)).dt.days
lab["recency_bucket"] = pd.cut(lab["recency_days"], [-1, 7, 14, 30, 60, 90, 180, 1000])
inactive_rate = lab.groupby("recency_bucket", observed=True)["Opportunity"].apply(lambda s: (s == "Inactivity").mean())

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(inactive_rate.index.astype(str), inactive_rate.values, color=ORANGE, width=0.7)
ax.set_title("Share Inactivity next quarter by days since last visit", loc="left")
ax.set_ylim(0, 1)
inactive_rate.round(3)

## 7. Takeaways

1. Last quarter behaviour is a strong baseline for next quarter, the model has to beat "same as before".
2. Recency is the key driver of Inactivity: over half of customers not seen for 60+ days are inactive next quarter, versus about 3% of those seen in the last week.
3. Targets have lots of zeros, then a skewed tail. The log transform tames the tail, so training straight on the transformed target is the right move.
4. The label mix shifts over time and the test quarter includes December, so we validate on the most recent quarter and keep the cutoff month as a feature.
5. Category history (what a customer already buys) separates "growth" from "adoption", so per category spend features belong in the model.